# 05 - Analysis And Reporting

This notebook loads the detection dataset, computes model/category/persona effects, saves report tables and plots to Google Drive, and logs them to W&B.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

DEFAULT_REPO_URL = "https://github.com/ritwikraha/AutoRegressive-Bhasha.git"

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/AutoRegressive-Bhasha/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", DEFAULT_REPO_URL)
    target = Path("/content/AutoRegressive-Bhasha")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target)], check=True)

    cloned_candidates = [target / "empty-negations", target]
    for candidate in cloned_candidates:
        if (candidate / "src/ocn").exists():
            return candidate

    raise FileNotFoundError(
        f"Cloned {repo_url}, but could not find src/ocn. "
        "Set OCN_REPO_URL to a repository containing empty-negations/src/ocn."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
import wandb
from datasets import load_dataset

from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, save_dataframe, utc_timestamp
from ocn.metrics import detection_summary, grouped_ocn_rates, top_patterns

paths = make_colab_paths()
config = json.loads((paths.project_root / "ocn_colab_config.json").read_text())
_ = login_huggingface("HF_WRITE_ACCESS")
EXPERIMENT_ID = "main_gemma4_qwen35"
ANALYSIS_RUN_ID = utc_timestamp()
MAIN_DETECTION_REPO = config.get(
    "hf_main_detection_repo",
    f"{config['hf_owner']}/ocn-empty-negations-detection-main-gemma4-qwen35",
)
analysis_config = {
    **config,
    "experiment_id": EXPERIMENT_ID,
    "analysis_run_id": ANALYSIS_RUN_ID,
    "source_repo": MAIN_DETECTION_REPO,
}
run = login_wandb(
    project="ocn-empty-negations",
    name=f"analysis-{EXPERIMENT_ID}-{ANALYSIS_RUN_ID}",
    config=analysis_config,
)
sns.set_theme(style="whitegrid")

In [ ]:
df = load_dataset(MAIN_DETECTION_REPO, split="train").to_pandas()
summary = detection_summary(df)
model_rates = grouped_ocn_rates(df, ["model_id", "model_stage", "decoding"])
prompt_rates = grouped_ocn_rates(df, ["category", "variant", "persona"])
save_dataframe(model_rates, Path(config["drive_data_root"]) / "report_model_rates_main_gemma4_qwen35.csv")
save_dataframe(prompt_rates, Path(config["drive_data_root"]) / "report_prompt_rates_main_gemma4_qwen35.csv")
summary

In [ ]:
regression_df = df.copy()
regression_df["has_ocn_int"] = regression_df["has_ocn"].astype(int)
formula = "has_ocn_int ~ C(model_stage) + C(model_family) + C(decoding) + C(variant) + C(persona) + length_target"
model = smf.logit(formula, data=regression_df).fit(disp=False)
report_path = Path(config["drive_data_root"]) / "logit_model_summary_main_gemma4_qwen35.txt"
report_path.write_text(model.summary().as_text(), encoding="utf-8")
print(model.summary())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.barplot(data=model_rates, y="model_id", x="ocn_rate", hue="decoding", ax=axes[0, 0])
axes[0, 0].set_title("OCN rate by model and decoding")
axes[0, 0].set_xlim(0, 1)

variant_rates = grouped_ocn_rates(df, ["variant"])
sns.barplot(data=variant_rates, y="variant", x="ocn_rate", ax=axes[0, 1], color="#f58518")
axes[0, 1].set_title("OCN rate by prompt variant")
axes[0, 1].set_xlim(0, 1)

persona_rates = grouped_ocn_rates(df, ["persona"])
sns.barplot(data=persona_rates, y="persona", x="ocn_rate", ax=axes[1, 0], color="#54a24b")
axes[1, 0].set_title("OCN rate by persona")
axes[1, 0].set_xlim(0, 1)

patterns = top_patterns(df, 12)
sns.barplot(data=patterns, y="pattern", x="count", ax=axes[1, 1], color="#b279a2")
axes[1, 1].set_title("Top detector patterns")
plt.tight_layout()

fig_path = Path(config["drive_figure_root"]) / "05_analysis_dashboard_main_gemma4_qwen35.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
wandb.log({
    **summary.to_dict(),
    "analysis_dashboard": wandb.Image(str(fig_path)),
    "model_rates": wandb.Table(dataframe=model_rates),
    "prompt_rates": wandb.Table(dataframe=prompt_rates),
    "logit_summary": model.summary().as_text(),
})
run.finish()
fig_path